In [8]:
# Only if XGBoost isn’t installed in this env
!pip install xgboost joblib pandas scikit-learn


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, roc_auc_score
import joblib


In [10]:
# Load your monthly clinical scores CSV
df = pd.read_csv('../data/monthly_clinical_scores.csv')

# Features: MBI subscales + BDI + GAD + MHC
X = df[['EE', 'DP', 'PA', 'BDI', 'GAD', 'MHC']]

# Binary target: “high burnout risk” if Emotional Exhaustion ≥ 27
y = (df['EE'] >= 27).astype(int)

# Quick sanity check on class balance
print("Full-data class distribution:\n", y.value_counts(), "\n")


Full-data class distribution:
 EE
1    1
0    1
Name: count, dtype: int64 



In [11]:
# Determine whether to stratify or not
if y.value_counts().min() >= 2:
    strat_arg = y
    print("✅ Using stratified split\n")
else:
    strat_arg = None
    print("⚠️ Small dataset — skipping stratification\n")

# Perform the split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=strat_arg
)

# Verify resulting distributions
print("Train set:\n", y_train.value_counts())
print("Test  set:\n", y_test.value_counts(), "\n")



⚠️ Small dataset — skipping stratification

Train set:
 EE
1    1
Name: count, dtype: int64
Test  set:
 EE
0    1
Name: count, dtype: int64 



In [12]:
# Instantiate and fit XGBoost regressor
model = XGBRegressor(use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)



c:\Users\kumar\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:183: UserWarning: [02:24:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='logloss',
             feature_types=None, feature_weights=None, gamma=None,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, ...)

In [13]:
preds = model.predict(X_test)

# Regression metric
print("RMSE:", mean_squared_error(y_test, preds, squared=False))

# Compute ROC-AUC only if both classes are present
if len(y_test.unique()) == 2:
    print("ROC-AUC:", roc_auc_score(y_test, preds))
else:
    print("ROC-AUC: skipped (only one class in test set)")


RMSE: 1.0
ROC-AUC: skipped (only one class in test set)


c:\Users\kumar\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [14]:
joblib.dump(model, '../models/burnout_model.pkl')
print("✅ Model saved to ../models/burnout_model.pkl")


✅ Model saved to ../models/burnout_model.pkl
